# Lecture 9


## Core Vision Tasks Overview

Image classification puts a single global label on an image. Semantic segmentation goes further by classifying every pixel into a category, but it ignores individual object instances. Object detection takes it up a notch by predicting bounding boxes, defined by coordinates (x, y, w, h), and category labels for multiple objects. Instance segmentation combines these by detecting individual objects and predicting a specific pixel-level mask for each one.

Softmax cross-entropy loss never hits absolute zero in practice. Softmax turns raw scores into a probability distribution where no class probability can ever be exactly 0 or 1.


## Semantic Segmentation & Upsampling

The old-school way to do semantic segmentation was sliding a window across the image, cropping small patches around each pixel, and running a standard CNN on them. This is way too slow because you end up repeating the same feature extraction work for overlapping patches and you lose the global context of the image.

Fully Convolutional Networks (FCNs) fix this by ditching the fully connected layers and running the whole image through convolutions to get a spatial score map of shape C x H x W. Doing this at full resolution is heavy on memory, so most designs downsample the features first to catch high-level patterns and then upsample them back to the original size.

For upsampling, you can use simple methods like Nearest Neighbor or Bed of Nails. A more clever approach is Max Unpooling. This remembers the exact spots where the max values were during downsampling and puts those values back in the same places during upsampling, filling the rest with zeros.

Transposed convolution is a way to learn how to upsample. Instead of computing dot products over patches, each input pixel multiplies a K x K filter, and the model writes those weighted values into the output map, summing them up wherever they overlap.


question to revisit: what happens here when the input size isn't divisible?


this derivation felt shaky, redo it on paper before moving on


connection: this shows up again in the architectures lecture, remember it


In [ ]:
import torch.nn as nn

# Learnable 2x spatial upsampling using transposed convolution
upsample = nn.ConvTranspose2d(
    in_channels=256, out_channels=128, kernel_size=3, stride=2, padding=1, output_padding=1
)

## Two-Stage Object Detection (R-CNN Family)

Detecting a single object uses a combined loss function for classification and L2 regression on the bounding box (x, y, w, h). You cannot use fixed-size regression outputs for multiple objects because you never know how many are in an image.

The original R-CNN generated about 2,000 region proposals per image using Selective Search on the CPU. It warped every crop to 224 x 224, ran each one through a CNN separately, and used linear SVMs to classify them. It was painfully slow because it required 2,000 separate forward passes for one image.

Fast R-CNN speeds things up by running the whole image through the backbone once. Then, it uses RoI Pooling to project the proposals onto the feature map. It snaps the coordinates to the grid, max-pools each region to a fixed size (like 7 x 7), and sends those to the final classification and regression heads.

RoI Pooling has a flaw where rounding coordinates to fit the grid messes up the spatial alignment. RoI Align fixes this by ditching the grid snapping and using bilinear interpolation to sample features at precise points:

f_xy = sum(f_ij * max(0, 1 - |x - x_i|) * max(0, 1 - |y - y_j|)) for i,j in 1..2

Faster R-CNN replaces the slow CPU-based Selective Search with a Region Proposal Network (RPN). The RPN slides over the feature map, checks K anchor boxes of different scales and shapes, and predicts object scores and box offsets. The whole system is trained together using four joint losses.


this derivation felt shaky, redo it on paper before moving on


connection: this shows up again in the architectures lecture, remember it


got tripped up by the stride math here, worth a second pass


intuition check: explain this out loud without looking at the slides


the diagram for this one in the slides made it click finally


reminder: run the code cell above and tweak the filter size


kept confusing these two terms, write them out explicitly next time


skimmed this too fast the first time, the math is simpler than it looked


side note: tried rebuilding this part from memory and mixed up the indexing


## Single-Stage Object Detectors

Models like YOLO, SSD, and RetinaNet skip the region proposal step to get real-time performance. They divide the image into a grid (S x S), and anchor boxes centered at each cell regress the final coordinates and probabilities directly. They are much faster than two-stage detectors but have historically struggled a bit more with small objects.
